# 01 | Data preparation in Google Earth Engine

Builds a cloud-free 2025 Sentinel-2 composite for Kinabatangan, derives NDVI/EVI, creates forest-loss labels from Hansen (2019 to 2023), excludes protected reserves and exports a six-band feature stack plus label to Google Drive.

Set `EE_PROJECT` to your own Earth Engine project ID.

In [ ]:
!pip install -q earthengine-api
import ee
ee.Authenticate()
EE_PROJECT = "your-ee-project-id"
ee.Initialize(project=EE_PROJECT)

## Area of interest

In [ ]:
gaul = ee.FeatureCollection("FAO/GAUL/2015/level2")
aoi = gaul.filter(ee.Filter.eq("ADM2_NAME", "Kinabatangan")).geometry()
print("District area (km2):", aoi.area().divide(1e6).getInfo())

## Exclude protected reserves (WDPA)

In [ ]:
wdpa = ee.FeatureCollection("WCMC/WDPA/current/polygons").filterBounds(aoi)
reserves = wdpa.geometry()
aoi_clean = aoi.difference(reserves, 1)

total = aoi.area().divide(1e6).getInfo()
clean = aoi_clean.area().divide(1e6).getInfo()
print(f"Reserves cover {100 * (1 - clean / total):.1f}% of the district")

## Cloud-free Sentinel-2 composite and indices

In [ ]:
def mask_s2(img):
    qa = img.select("QA60")
    cloud = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return img.updateMask(cloud).divide(10000)


def get_clean_composite(start, end, region):
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterBounds(region)
           .filterDate(start, end)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
           .map(mask_s2))
    print("Images used:", col.size().getInfo())
    return col.median().clip(region)


def add_indices(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    evi = img.expression(
        "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
        {"NIR": img.select("B8"), "RED": img.select("B4"), "BLUE": img.select("B2")},
    ).rename("EVI")
    return img.addBands([ndvi, evi])


comp_2025 = add_indices(get_clean_composite("2025-01-01", "2025-12-31", aoi_clean))

## Labels from Hansen Global Forest Change

`lossyear` is stored as years since 2000, so 19 to 23 means 2019 to 2023. Only pixels that were forest in 2000 can be labelled as loss.

In [ ]:
hansen = ee.Image("UMD/hansen/global_forest_change_2023_v1_11")
loss_year = hansen.select("lossyear")
tree_cover = hansen.select("treecover2000")

# Loss per year, as a sanity check
for yr in range(19, 24):
    area = (ee.Image.pixelArea().updateMask(loss_year.eq(yr)).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi_clean, scale=30, maxPixels=1e10)
        .get("area").getInfo())
    print(2000 + yr, round(area / 1e6, 1), "km2")

loss_mask = tree_cover.gte(30)
label = (loss_year.gte(19).And(loss_year.lte(23)).And(loss_mask)).rename("label")

## Stack and export

Earth Engine requires every band in one export to have the same type, hence `.toFloat()`. The label is reprojected onto the 10 m Sentinel-2 grid.

In [ ]:
bands = comp_2025.select(["B2", "B3", "B4", "B8", "NDVI", "EVI"])
ref_proj = comp_2025.select("B4").projection()
label_10m = label.reproject(ref_proj).toFloat()

stack = bands.toFloat().addBands(label_10m).clip(aoi_clean)

task = ee.batch.Export.image.toDrive(
    image=stack,
    description="kinabatangan_stack_2025_v2",
    folder="forest_loss_project",
    fileNamePrefix="kinabatangan_stack_2025_v2",
    region=aoi_clean,
    scale=10,
    crs="EPSG:4326",
    maxPixels=1e10,
)
task.start()
print("Export started. Check the Tasks tab or run task.status()")